# Headless Quick View — using Quantum Metal without the Qt GUI

This notebook shows how to design and **view** components in Quantum Metal **without launching the Qt desktop GUI**. The full API works headlessly in a plain Python interpreter or in cloud notebook environments like Jupyter, Colab, and Binder where Qt isn't available or wanted.

## What you'll learn

1. How to build a design programmatically — same API as the GUI workflow
2. How to use `qm.view(design)` to render a design to a matplotlib figure inline
3. How to customise the view: filter components, hide layers, render into existing axes
4. How to save the rendered design to a file

## When to use this workflow

- You're running on a server / cloud notebook (Colab, Binder, JupyterHub) where PySide6 isn't installed.
- You're scripting a parameter sweep and just want to verify the geometry looks right.
- You're embedding Quantum Metal in a larger pipeline (CI, automated optimization, papers/figures).

If you want the **interactive editor** with click-to-select components, dockable panels, and live option editing, use `qm.MetalGUI(design)` instead. That requires a local installation with PySide6.

## Setup

We force matplotlib's `Agg` backend before any plot calls so the notebook works on headless runners. In a normal Jupyter session you can skip this — Jupyter picks an inline-friendly backend automatically.

In [ ]:
import matplotlib

matplotlib.use("Agg")  # only needed when not in Jupyter

import qiskit_metal as qm
from qiskit_metal import Dict
from qiskit_metal.designs import DesignPlanar
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

## Build a small design programmatically

Same API as the GUI workflow — no Qt window is created.

In [ ]:
design = DesignPlanar()

# A transmon qubit with one coupling pad
TransmonPocket(
    design,
    "Q1",
    options=Dict(
        pos_x="0mm",
        pos_y="0mm",
        connection_pads=Dict(
            a=Dict(loc_W="+1", loc_H="+1"),
        ),
    ),
)

# A ground termination next to it
OpenToGround(
    design,
    "G1",
    options=Dict(pos_x="1mm", pos_y="0mm"),
)

print("Components in design:", list(design.components.keys()))

## View the design

`qm.view(design)` returns a `matplotlib.figure.Figure`. In Jupyter it will display inline; you can also save it or further customise it like any other matplotlib figure.

In [ ]:
fig = qm.view(design)
fig

## Iterating on the design

Changing a component's options is the same workflow as in the GUI — but you get a new `Figure` instead of an updated Qt canvas. Component objects are accessible by name via `design.components`.

In [ ]:
# Change Q1's pocket dimensions and re-render — no Qt window
design.components["Q1"].options.pocket_width = "800um"
design.components["Q1"].options.pocket_height = "500um"
design.rebuild()

fig = qm.view(design, title="After resizing Q1")
fig

## Customising the view

### Filter to specific components

In [ ]:
fig = qm.view(design, components=["Q1"], title="Q1 only")
fig

### Custom figure size

In [ ]:
fig = qm.view(design, figsize=(6, 4))
fig

### Side-by-side in a multi-panel figure

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
qm.view(design, ax=axes[0], title="Full design")
qm.view(design, ax=axes[1], components=["Q1"], title="Just Q1")
fig

## Save to a file

Any `matplotlib.figure.Figure` method works — `savefig` to PNG/PDF/SVG, embed in a paper, etc.

In [ ]:
fig = qm.view(design)
fig.savefig("my_design.png", dpi=150, bbox_inches="tight")
print("Saved my_design.png")

## What about HFSS / Q3D / GDS?

All non-Qt renderers work headlessly too:

- **GDS export** — `QGDSRenderer(design).export_to_gds("out.gds")` works without Qt.
- **HFSS / Q3D** — the *renderer* doesn't require Qt, but it does require Ansys AEDT installed on the machine running the script. On a cloud notebook without Ansys, you can still build the design and export GDS / visualise with `qm.view`; you just can't run the live solve.
- **Analyses** — every module in `qiskit_metal.analyses` is pure Python (numpy / scipy / qutip) and works headlessly.

## Next steps

- See `docs/headless-usage.rst` for the full reference.
- See the [1.2 Quick start](./1.2%20Quick%20start.ipynb) notebook for a tour of the GUI-driven workflow.
- For interactive pan/zoom in Jupyter without Qt, install `ipympl` and run `%matplotlib widget` at the top of your notebook.

## What's next

* Browse [1.1 Bird's eye view](./1.1%20Bird%27s%20eye%20view%20of%20Qiskit%20Metal.ipynb) and [1.2 Quick start](./1.2%20Quick%20start.ipynb) — they use `MetalGUI` for visualisation but every concept transfers; just replace `gui.rebuild()` with `qm.view(design)`. The GUI-screenshots in those tutorials give you a sense of what the desktop experience looks like.
* See the **2 From components to chip** folder for multi-qubit designs and CPW routing. The same `qm.view(design)` call works for any of them — pass `figsize=(12, 12)` for chip-scale views.
* For programmatic GDS export (no Qt or Ansys required), use the `QGDSRenderer`:

  ```python
  from qiskit_metal.renderers.renderer_gds.gds_renderer import QGDSRenderer
  gds = QGDSRenderer(design)
  gds.export_to_gds("my_chip.gds")
  ```

* Read the full reference in [`docs/headless-usage.rst`](../../docs/headless-usage.rst) for all of `qm.view`'s options, the `[gui]`/`[ansys]`/`[fem]` install extras, and the roadmap for a future richer interactive viewer.